# FER-2013 Emotion Detection — CNN Training (Colab)

This notebook downloads **msambare/fer2013**, builds a CNN, trains, and exports `emotion_model.h5`.

In [ ]:
!pip -q install kaggle
from google.colab import files
print('Upload your kaggle.json when prompted...')
files.upload();
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d fer2013
print('Dataset ready at ./fer2013')


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_dir = 'fer2013/train'
test_dir = 'fer2013/test'
train_datagen = ImageDataGenerator(rescale=1./255,
    rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')
test_datagen = ImageDataGenerator(rescale=1./255)
train_gen = train_datagen.flow_from_directory(train_dir, target_size=(48,48),
    batch_size=64, color_mode='grayscale', class_mode='categorical')
test_gen = test_datagen.flow_from_directory(test_dir, target_size=(48,48),
    batch_size=64, color_mode='grayscale', class_mode='categorical')
num_classes = 7


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
model = Sequential([
  Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),
  BatchNormalization(),
  MaxPooling2D(2,2),
  Dropout(0.25),
  Conv2D(64, (3,3), activation='relu'),
  BatchNormalization(),
  MaxPooling2D(2,2),
  Dropout(0.25),
  Conv2D(128, (3,3), activation='relu'),
  BatchNormalization(),
  MaxPooling2D(2,2),
  Dropout(0.25),
  Flatten(),
  Dense(256, activation='relu'),
  Dropout(0.5),
  Dense(7, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
EPOCHS = 30
history = model.fit(train_gen, epochs=EPOCHS, validation_data=test_gen)


In [ ]:
model.save('emotion_model.h5')
print('Saved as emotion_model.h5')


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
test_gen.reset()
preds = model.predict(test_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
print(classification_report(test_gen.classes, y_pred, target_names=list(test_gen.class_indices.keys())))
